# EDA cleaning: preparing NOAA and EAGLE-I for joining

This notebook reviews **full-dataset** cleaning outputs from `clean_datasets.py`. Raw CSVs are unchanged; all rows are retained. The goal is reliable identifiers and timestamps before choosing an outage target and joining storm intervals to observations.

The pipeline uses DuckDB to write compressed Parquet without loading the 249 million EAGLE-I rows into pandas. Pandas is used here only for small report tables and previews.

In [1]:
from pathlib import Path
import json
import sys
import subprocess
import duckdb
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
CLEAN = ROOT / "data" / "03_cleaned"
REPORTS = ROOT / "reports" / "quality"
pd.set_option("display.max_columns", None)

# Each Parquet output has a .meta.json sidecar (cleaning version + query hash).
# The script reuses an output only when both match, so rerunning after a rule change rebuilds what changed.
RUN_CLEANING = False
if RUN_CLEANING:
    subprocess.run([sys.executable, str(ROOT / "pipeline" / "04_clean_datasets.py")], check=True)

manifest = json.loads((CLEAN / "cleaning_manifest.json").read_text())
con = duckdb.connect()
con.execute("SET TimeZone='UTC'")

def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"

for view, file in [("storms", "storm_details.parquet"), ("outages", "eaglei.parquet"),
                   ("locations", "storm_locations.parquet"), ("fatalities", "storm_fatalities.parquet"),
                   ("join_audit", "storm_join_audit.parquet")]:
    con.execute(f"CREATE VIEW {view} AS SELECT * FROM read_parquet({sql_string(CLEAN / file)})")

def report(filename):
    with pd.option_context("display.max_rows", 100):
        display(pd.read_csv(REPORTS / filename, dtype={"county_fips": "string"}))

print(f"Cleaning version {manifest['cleaning_version']}")
display(pd.Series(manifest["rows"], name="rows").to_frame())
display(pd.Series(manifest["eaglei_row_accounting"], name="eaglei_rows").to_frame())

Cleaning version 3


,rows
storms,771302
locations,690006
fatalities,11399
outages_raw,249316543
outages,249076978


,eaglei_rows
raw_rows,249316543
dec31_rows,262905
ct_region_rows,105828
ct_legacy_after_cutover_rows,73
ct_allocated_rows,129241
final_rows,249076978


## Cleaning decisions

| Area | Rule |
|---|---|
| County identifiers | Five-character strings, including leading zeros; numeric formatting must be valid before padding. |
| NOAA county events | Combine two-character state code and three-character county code only when `CZ_TYPE == C`. |
| Territory codes | Source data identifies Puerto Rico as 99 and Virgin Islands as 96; use Census state prefixes 72 and 78. Other nonstandard territories remain unresolved. |
| Alaska county-labeled events | Withhold county keys for geographic review: source codes include forecast-area-style identifiers. |
| NOAA zone events | Preserve zone identifiers and flag `zone_crosswalk_required`; never concatenate a zone code into a county key. |
| Times | Parse NOAA local timestamps and subtract the explicit recorded UTC offset. Treat EAGLE-I observation times as UTC. |
| Daylight labels | Honor the recorded numeric offset but flag daylight labels for review because NOAA guidance specifies standard time. |
| Outage target | Keep `customers_out`; it is the only outage measure in EAGLE-I. Preserve nulls with `target_observed=False`. |
| Zero-outage rows | 2014-2021 omit zero rows; 2022+ record them. `recorded_zero` flags them so the convention can be harmonized before a target is defined. |
| Total customers | Dropped: populated only in 2024. |
| Dec 31 | EAGLE-I observations on Dec 31 (UTC) are dropped for every year because 2014-2021 files end at Dec 31 00:00 UTC. Storms touching that day are excluded from join preparation. |
| Long-running events | Drought events and events longer than 72 hours are excluded from join preparation. They stay in `storm_details` with `join_exclusion` set. |
| Connecticut | EAGLE-I moved to planning regions on 2025-05-29. Regions are allocated back to the 8 legacy counties by 2020 town population (`ct_allocated=True`). |
| Untracked counties | NOAA county codes never observed in EAGLE-I are listed in `untracked_counties.csv` for exclusion from the final model. |
| Versioning | Every Parquet output has a `.meta.json` sidecar with the cleaning version and build hash. Stale outputs are rebuilt automatically. |
| Supporting NOAA tables | Keep separate, linked by numeric event ID. Aggregate one-to-many location/fatality records before any merge into a modeling table. |
| Modeling split | Reserve 2025; mark 2024 as a validation candidate. Final temporal split must also account for event windows crossing year boundaries. |

These are join-preparation tables, **not approved feature matrices**. Outcome fields and retrospective descriptions remain available for auditing; their presence does not authorize their use as predictors.

## NOAA: standardized identifiers and UTC times

In [2]:
display(con.execute("""SELECT EVENT_ID, STATE, STATE_FIPS, CZ_TYPE, CZ_FIPS,
    state_fips_clean, cz_code_clean, county_fips, CZ_TIMEZONE,
    begin_local, begin_utc, end_utc, geography_status, time_status
    FROM storms LIMIT 10""").df())
display(pd.Series(manifest["geography_status_counts"], name="events").to_frame())

,EVENT_ID,STATE,STATE_FIPS,CZ_TYPE,CZ_FIPS,state_fips_clean,cz_code_clean,county_fips,CZ_TIMEZONE,begin_local,begin_utc,end_utc,geography_status,time_status
0,503953,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-02-18 10:00:00,2014-02-18 15:00:00+00:00,2014-02-19 01:00:00+00:00,zone_crosswalk_required,ok
1,504065,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-02-05 03:00:00,2014-02-05 08:00:00+00:00,2014-02-06 04:00:00+00:00,zone_crosswalk_required,ok
2,494521,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-01-18 10:00:00,2014-01-18 15:00:00+00:00,2014-01-19 12:00:00+00:00,zone_crosswalk_required,ok
3,549746,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-11-26 10:00:00,2014-11-26 15:00:00+00:00,2014-11-27 15:00:00+00:00,zone_crosswalk_required,ok
4,503982,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-02-13 06:30:00,2014-02-13 11:30:00+00:00,2014-02-14 13:00:00+00:00,zone_crosswalk_required,ok
5,512089,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-04-24 10:14:00,2014-04-24 15:14:00+00:00,2014-04-24 18:16:00+00:00,zone_crosswalk_required,ok
6,495626,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-01-02 05:00:00,2014-01-02 10:00:00+00:00,2014-01-03 15:00:00+00:00,zone_crosswalk_required,ok
7,512096,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-04-15 12:55:00,2014-04-15 17:55:00+00:00,2014-04-15 17:55:00+00:00,zone_crosswalk_required,ok
8,507059,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-03-26 13:42:00,2014-03-26 18:42:00+00:00,2014-03-26 18:42:00+00:00,zone_crosswalk_required,ok
9,549721,NEW HAMPSHIRE,33,Z,12,33,012,None,EST-5,2014-11-02 11:40:00,2014-11-02 16:40:00+00:00,2014-11-02 18:10:00+00:00,zone_crosswalk_required,ok


,events
zone_crosswalk_required,330675
county_code_candidate,440166
alaska_geography_review,261
territory_or_identifier_review,200


### Timezone review

In [3]:
report("timezone_review.csv")

,CZ_TIMEZONE,time_status,events
0,CST-6,ok,355148
1,EST-5,ok,266069
2,MST-7,ok,96042
3,PST-8,ok,40146
4,HST-10,ok,6843
5,AST-4,ok,3262
6,AKST-9,ok,2867
7,SST-11,ok,293
8,PDT-7,explicit_daylight_offset_review,290
9,GST10,ok,275


## EAGLE-I: normalized county/time keys and target flags

The canonical observation key is `(county_fips, observation_utc)`. County names are descriptive fields, not join keys. Raw FIPS and timestamp text are retained for comparison.

In [4]:
display(con.execute("""SELECT fips_code_raw, county_fips, county, state,
    timestamp_raw, observation_utc, customers_out,
    join_key_valid, target_observed, recorded_zero, ct_allocated, split_role
    FROM outages LIMIT 10""").df())
report("eaglei_cleaning_by_year.csv")

,fips_code_raw,county_fips,county,state,timestamp_raw,observation_utc,customers_out,join_key_valid,target_observed,recorded_zero,ct_allocated,split_role
0,01037,01037,Coosa,Alabama,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,12,True,True,False,False,training_candidate
1,01051,01051,Elmore,Alabama,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,7,True,True,False,False,training_candidate
2,01109,01109,Pike,Alabama,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,1,True,True,False,False,training_candidate
3,01121,01121,Talladega,Alabama,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,31,True,True,False,False,training_candidate
4,04017,04017,Navajo,Arizona,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,1,True,True,False,False,training_candidate
5,05009,05009,Boone,Arkansas,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,3,True,True,False,False,training_candidate
6,05119,05119,Pulaski,Arkansas,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,1,True,True,False,False,training_candidate
7,06029,06029,Kern,California,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,30,True,True,False,False,training_candidate
8,06037,06037,Los Angeles,California,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,1555,True,True,False,False,training_candidate
9,06065,06065,Riverside,California,2014-11-01 04:00:00,2014-11-01 04:00:00+00:00,2,True,True,False,False,training_candidate


,source_year,rows,dec31_rows_dropped,invalid_join_keys,unknown_or_invalid_target,recorded_zero_rows,ct_allocated_rows,source_year_mismatches,first_utc,last_utc
0,2014,1689184,276,0,0,0,0,0,2014-11-01 04:00:00+00,2014-12-30 23:45:00+00
1,2015,12938705,452,0,0,0,0,0,2015-01-01 00:00:00+00,2015-12-30 23:45:00+00
2,2016,13305726,298,0,0,0,0,0,2016-01-01 00:00:00+00,2016-12-30 23:45:00+00
3,2017,15077945,419,0,162588,0,0,0,2017-01-01 00:00:00+00,2017-12-30 23:45:00+00
4,2018,21776302,504,0,2027705,0,0,0,2018-01-01 00:00:00+00,2018-12-30 23:45:00+00
5,2019,24073175,947,0,1366540,0,0,0,2019-01-01 00:00:00+00,2019-12-30 23:45:00+00
6,2020,25544859,658,0,2101440,0,0,0,2020-01-01 00:00:00+00,2020-12-30 23:45:00+00
7,2021,24825488,614,0,1104718,0,0,0,2021-01-01 00:00:00+00,2021-12-30 23:45:00+00
8,2022,25739509,56956,0,0,1276103,0,0,2022-01-01 00:00:00+00,2022-12-30 23:45:00+00
9,2023,26059090,41961,0,0,1305236,0,0,2023-01-01 00:00:00+00,2023-12-30 23:45:00+00


### Zero-outage convention changes in 2022

Before 2022 a county with no outages has no row; from 2022 a zero is recorded. Row counts, coverage, and any occurrence target are not comparable across that break until it is harmonized. Two consistent options: drop `recorded_zero` rows from 2022+ (all years then mean "row = outage reported"), or build targets on a fixed county/time grid where absence is treated as unknown, not zero.

In [5]:
display(con.execute("""SELECT source_year, count(*) AS rows,
    count(*) FILTER (WHERE recorded_zero) AS recorded_zero_rows,
    round(100.0 * count(*) FILTER (WHERE recorded_zero) / count(*), 1) AS pct_zero
    FROM outages GROUP BY 1 ORDER BY 1""").df())

,source_year,rows,recorded_zero_rows,pct_zero
0,2014,1689184,0,0.0
1,2015,12938705,0,0.0
2,2016,13305726,0,0.0
3,2017,15077945,0,0.0
4,2018,21776302,0,0.0
5,2019,24073175,0,0.0
6,2020,25544859,0,0.0
7,2021,24825488,0,0.0
8,2022,25739509,1276103,5.0
9,2023,26059090,1305236,5.0


### Connecticut planning-region consolidation

Weights come from `pipeline/03_build_ct_crosswalk.py` (Census town crosswalk and 2020 town populations). An allocated value is null when any contributing region reported a null; `ct_allocation_complete` is False when a region was absent or null at that timestamp.

In [6]:
display(pd.read_csv(ROOT / "data" / "reference" / "ct_region_to_county_weights.csv", dtype={"region_fips": "string", "county_fips": "string"}))
display(con.execute("""SELECT county_fips, any_value(county) AS county, ct_allocated,
    min(observation_utc) AS first_utc, max(observation_utc) AS last_utc, count(*) AS rows,
    count(*) FILTER (WHERE ct_allocated AND NOT ct_allocation_complete) AS incomplete_rows
    FROM outages WHERE source_year = 2025 AND county_fips LIKE '09%'
    GROUP BY county_fips, ct_allocated ORDER BY county_fips, ct_allocated""").df())

,region_fips,county_fips,population_2020,weight
0,09110,09003,827269,0.847373
1,09110,09013,149006,0.152627
2,09120,09001,325774,1.000000
3,09130,09007,164250,0.942746
4,09130,09011,9975,0.057254
5,09140,09001,40883,0.090763
6,09140,09003,60847,0.135085
7,09140,09005,54332,0.120621
8,09140,09009,294373,0.653530
9,09150,09011,2569,0.026947


,county_fips,county,ct_allocated,first_utc,last_utc,rows,incomplete_rows
0,09001,Fairfield,False,2025-01-01 01:15:00+00:00,2025-05-29 16:45:00+00:00,10194,0
1,09001,Fairfield,True,2025-05-29 17:00:00+00:00,2025-12-30 23:45:00+00:00,18721,13307
2,09003,Hartford,False,2025-01-01 02:00:00+00:00,2025-05-29 16:45:00+00:00,10057,0
3,09003,Hartford,True,2025-05-29 17:00:00+00:00,2025-12-30 23:45:00+00:00,18854,10906
4,09005,Litchfield,False,2025-01-01 00:45:00+00:00,2025-05-29 16:45:00+00:00,8007,0
5,09005,Litchfield,True,2025-05-29 17:00:00+00:00,2025-12-30 23:45:00+00:00,18880,10866
6,09007,Middlesex,False,2025-01-01 00:00:00+00:00,2025-05-29 16:30:00+00:00,5376,0
7,09007,Middlesex,True,2025-05-29 19:45:00+00:00,2025-12-30 23:45:00+00:00,10004,0
8,09009,New Haven,False,2025-01-01 02:15:00+00:00,2025-05-29 16:45:00+00:00,8678,0
9,09009,New Haven,True,2025-05-29 17:00:00+00:00,2025-12-30 23:45:00+00:00,15714,7724


### Identifier and timestamp normalization collisions

In [7]:
report("fips_normalization_collisions.csv")
report("timestamp_normalization_collisions.csv")

,county_fips,raw_forms


,observation_utc,raw_forms


## Storm events excluded from join preparation

`long_running_event`: Drought or longer than 72 hours. `overlaps_dec31_gap`: interval touches Dec 31 UTC, where EAGLE-I rows are dropped. Excluded events remain in `storm_details`; join readiness below covers only the remaining events.

In [8]:
display(pd.Series(manifest["storm_join_exclusions"], name="events").to_frame())
excl = pd.read_csv(REPORTS / "storm_exclusions.csv")
display(excl.groupby(["join_exclusion", "EVENT_TYPE"])["events"].sum().sort_values(ascending=False).head(15).to_frame())

,events
overlaps_dec31_gap,2118
long_running_event,54625


events
join_exclusion     EVENT_TYPE                     
long_running_event Drought                   37092
                   Flood                      5331
                   Excessive Heat             3245
                   Heat                       2925
                   Wildfire                   2196
                   High Surf                  1234
                   Extreme Cold/Wind Chill     585
overlaps_dec31_gap Winter Weather              421
long_running_event Winter Storm                396
                   Cold/Wind Chill             356
                   Heavy Snow                  309
overlaps_dec31_gap Heavy Snow                  302
                   Extreme Cold/Wind Chill     299
                   Winter Storm                239
                   Cold/Wind Chill             144

## Join readiness by year

`county_seen_in_eaglei` tests whether the cleaned county code appears anywhere in EAGLE-I. `overlaps_observation_envelope` only tests whether the storm interval intersects a county/year's first-to-last observation range. **Neither proves an outage observation exists at storm time or establishes complete coverage.**

County boundary changes are not yet harmonized across years. Review unmatched counties and use historically appropriate crosswalks before assigning zone events to counties.

In [9]:
report("storm_join_readiness.csv")

,SOURCE_YEAR,geography_status,time_status,events,county_seen_in_eaglei,overlaps_observation_envelope
0,2014,alaska_geography_review,ok,30,0,0
1,2014,county_code_candidate,ok,32226,31884,506
2,2014,territory_or_identifier_review,ok,2,0,0
3,2014,zone_crosswalk_required,ok,23134,0,0
4,2015,alaska_geography_review,ok,14,0,0
5,2015,county_code_candidate,ok,34686,34392,25913
6,2015,territory_or_identifier_review,ok,3,0,0
7,2015,zone_crosswalk_required,ok,20780,0,0
8,2016,alaska_geography_review,ok,7,0,0
9,2016,county_code_candidate,ok,33654,33322,25811


### Counties NOAA reports but EAGLE-I never observes

These are coverage gaps in EAGLE-I (for example five Tennessee counties and parts of Nebraska and Puerto Rico), not code errors. They are candidates to drop from the final modeling table; events there have no outage observations to label.

In [10]:
untracked = pd.read_csv(REPORTS / "untracked_counties.csv", dtype={"county_fips": "string"})
print(f"Untracked counties: {len(untracked):,} with {untracked.events.sum():,} eligible events")
display(untracked.groupby("STATE").agg(counties=("county_fips", "size"), events=("events", "sum")).sort_values("events", ascending=False))
display(untracked.head(20))

Untracked counties: 148 with 4,879 eligible events


,counties,events
STATE,,
PUERTO RICO,71,1782
NEBRASKA,14,1734
TENNESSEE,5,406
KANSAS,3,372
NEW MEXICO,2,251
COLORADO,1,139
MONTANA,1,70
GEORGIA,49,59
MINNESOTA,1,54


,STATE,county_fips,cz_name,events,first_year,last_year
0,NEBRASKA,31033,CHEYENNE,262,2014,2025
1,NEBRASKA,31141,PLATTE,205,2014,2025
2,NEBRASKA,31057,DUNDY,195,2014,2025
3,NEBRASKA,31029,CHASE,185,2014,2025
4,NEW MEXICO,35007,COLFAX,169,2014,2025
5,NEBRASKA,31129,NUCKOLLS,159,2014,2025
6,KANSAS,20187,STANTON,158,2014,2025
7,NEBRASKA,31181,WEBSTER,141,2014,2025
8,COLORADO,08095,PHILLIPS,139,2014,2025
9,PUERTO RICO,72103,NAGUABO,121,2014,2025


### Zone and territory review queue

In [11]:
review = pd.read_csv(REPORTS / "geography_review.csv", dtype="string")
print(f"Distinct source geographies needing review: {len(review):,}")
display(review.head(20))

Distinct source geographies needing review: 5,514


,STATE,STATE_FIPS,CZ_TYPE,CZ_FIPS,CZ_NAME,events
0,ATLANTIC SOUTH,87,Z,552,VOLUSIA-BREVARD COUNTY LINE TO SEBASTIAN INLET...,1596
1,WYOMING,56,Z,110,NORTH SNOWY RANGE FOOTHILLS,1503
2,ATLANTIC NORTH,88,Z,532,CHESAPEAKE BAY SANDY PT TO N BEACH MD,1102
3,ATLANTIC NORTH,88,Z,536,TIDAL POTOMAC INDIAN HD TO COBB IS MD,1013
4,WYOMING,56,Z,116,SOUTH LARAMIE RANGE,933
5,GULF OF MEXICO,85,Z,830,TAMPA BAY,729
6,WYOMING,56,Z,117,SOUTH LARAMIE RANGE FOOTHILLS,718
7,ATLANTIC NORTH,88,Z,537,TIDAL POTOMAC COBB IS MD TO SMITH PT VA,679
8,GULF OF MEXICO,85,Z,44,HAWK CHANNEL FROM WEST END OF SEVEN MILE BRIDG...,662
9,GULF OF MEXICO,85,Z,853,ENGLEWOOD TO TARPON SPRINGS FL OUT 20NM,585


### Supporting-table referential integrity

In [12]:
for kind in ["locations", "fatalities"]:
    orphan = pd.read_csv(REPORTS / f"{kind}_orphan_events.csv")
    print(f"{kind}: {len(orphan):,} rows with no matching storm detail event")
    if len(orphan):
        display(orphan.head())

locations: 0 rows with no matching storm detail event
fatalities: 0 rows with no matching storm detail event


## Safe next join design

1. Resolve zone-to-county mapping using boundaries/crosswalks appropriate to the event dates. Keep any one-to-many county mapping explicit.
2. Decide the prediction time, horizon, and target (for example maximum observed customers out during a defined window). NOAA records are retrospective; only features available at prediction time belong in a forecasting model.
3. Match observations on **county FIPS plus a UTC time window**, not equality of storm start time to a 15-minute observation and not a calendar-date-only merge.
4. Aggregate observations to the chosen event/county grain. Overlapping storms can share observations; do not interpret those matches as proof of causation or double-count them silently.
5. Keep matched-but-null targets and unmatched windows distinguishable. Do not invent zero-outage labels.
6. Split chronologically and handle windows/episodes spanning split boundaries. Fit imputation, scaling, and encoders on training data only.

### Source definitions

- [NOAA Storm Data instructions](https://www.weather.gov/media/directives/010_pdfs/pd01016005curr.pdf): event times use local standard time.
- [EAGLE-I source paper](https://www.nature.com/articles/s41597-024-03095-5): UTC timestamps; absent observations can reflect no outages or collection gaps. It documents 2014-2022; the same UTC convention is assumed for the appended 2023-2025 files.
- [Census Connecticut county-to-planning-region crosswalk](https://www2.census.gov/geo/docs/reference/ct_change/ct_cou_to_cousub_crosswalk.xlsx): town-level basis for the Connecticut allocation.
- [Census geographic code definitions](https://www.census.gov/library/reference/code-lists/ansi.2020.html): standardized state/county identifiers.

Full run details and limitations are in `data/03_cleaned/cleaning_manifest.json`.

In [13]:
con.close()